In [1]:
from pyspark.sql import functions as F

# Read GDP data from Bronze
gdp_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Bronze_LH.dbo.worldbank_gdp_raw"
)

print("Original records:", gdp_df.count())

# Clean and standardize GDP data
clean_gdp_df = (
    gdp_df
    .select(
        F.col("year").cast("int").alias("year"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.trim("country_name").alias("country_name"),
        F.trim("indicator").alias("indicator"),
        F.col("gdp_current_usd").cast("double").alias("gdp_current_usd")
    )
    .filter(F.col("year").isNotNull())
    .filter(F.col("country_code") == "USA")
    .filter(F.col("gdp_current_usd").isNotNull())
    .filter(F.col("gdp_current_usd") > 0)
    .dropDuplicates(["year", "country_code"])
    .orderBy("year")
)

# Save to Silver Lakehouse
(
    clean_gdp_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("worldbank_gdp_clean")
)

print("World Bank Silver data saved successfully!")
print("Clean records:", clean_gdp_df.count())

display(clean_gdp_df.orderBy(F.desc("year")))

StatementMeta(, decb9bd1-ba87-4676-8737-5d3cb0d189ba, 3, Finished, Available, Finished, False)

Original records: 66
World Bank Silver data saved successfully!
Clean records: 66


SynapseWidget(Synapse.DataFrame, 17191684-8a62-49e8-98c7-344f640c6d61)